### **Setup: Imports und Initialisierung**

In [1]:
import sys
import os
import pandas as pd

sys.path.append(os.path.abspath('..'))

from quinex_application.quinex_discussion import QuinexDiscussion
from quinex_application.quinex_visualisation import QuinexVisualisation

In [2]:
discussion = QuinexDiscussion()
visualisation = QuinexVisualisation()

### **Load Results**

In [3]:
df_quinex_metadata = discussion.load_df("../data/quinex/df_quinex_metadata.parquet")
df_quinex = discussion.load_df("../data/quinex/df_quinex.parquet")

📂 Loaded DataFrame from '../data/quinex/df_quinex_metadata.parquet' with 31633 rows
📂 Loaded DataFrame from '../data/quinex/df_quinex.parquet' with 3244165 rows


### **Filter data**

In [4]:
df = df_quinex.copy()
entity_regex_map = discussion.entity_regex_map
property_regex_map = discussion.property_regex_map

In [ ]:
property_regex_map = {"Emission": property_regex_map["Emission"]}
property_regex_map

In [5]:
df["entity_mapping"] = df["entity"].apply(lambda x: discussion.map_entity(x, entity_regex_map))
label_to_base = discussion.create_label_to_base(entity_regex_map)
df["entity_base_mapping"] = df["entity_mapping"].apply(lambda x: discussion.map_base_from_mapping(x, label_to_base))
df = df[df["entity_mapping"].notna()]

In [6]:
df["property_mapping"] = df["property"].apply(lambda x: discussion.map_entity(x, property_regex_map))
label_to_base = discussion.create_label_to_base(property_regex_map)
df["property_base_mapping"] = df["property_mapping"].apply(lambda x: discussion.map_base_from_mapping(x, label_to_base))
df = df[df["property_mapping"].notna()]

In [ ]:
df["property_mapping"].value_counts()

In [8]:
df.to_parquet("../data/visualisation_files/df_technologies.parquet")

### **Filter results**

In [ ]:
technologies_df = pd.read_parquet("../data/visualisation_files/df_technologies.parquet")

In [ ]:
technologies_df["property_mapping"].value_counts()

In [ ]:
technologies_df[technologies_df["property_mapping"] == "Capital Cost"]["property"].value_counts().head(50)

In [ ]:
technologies_df[(technologies_df["property_mapping"] == "CO2 emission") & (technologies_df["entity"] == "Berlin")].head(2)

### **Postprocess Units**

In [20]:
technologies_df = pd.read_parquet("../data/visualisation_files/df_technologies.parquet")
technologies_df = technologies_df[technologies_df["property_mapping"] == "Capital Cost"]
def dicts_to_tuples(units):
    return [(u.get('text', ''), u.get('exponent', None), u.get('unit_uri', ''), u.get('year', None)) 
            for u in units]

technologies_df['unit_normalized'] = technologies_df['unit_normalized'].apply(dicts_to_tuples)

In [21]:
print(technologies_df["unit_normalized"].value_counts())
print("--------------------------------------------------------------------------------")
exclude_values = [
    [(None, None, 'http://qudt.org/vocab/unit/UNITLESS', None)],
    [('%', 1.0, 'http://qudt.org/vocab/unit/PERCENT', None)],
    [(None, None, 'http://qudt.org/vocab/unit/UNITLESS', None)]   
]
technologies_df = technologies_df[~technologies_df["unit_normalized"].isin(exclude_values)]
print(technologies_df["unit_normalized"].value_counts().head(10))

unit_normalized
[(%, 1.0, http://qudt.org/vocab/unit/PERCENT, None)]                                                              484
[(None, None, http://qudt.org/vocab/unit/UNITLESS, None)]                                                         413
[($, 1.0, http://qudt.org/vocab/currency/USD, None)]                                                              378
[($, 1.0, http://qudt.org/vocab/currency/USD, None), (kW, -1.0, http://qudt.org/vocab/unit/KiloW, None)]          245
[(€/kW, 1.0, http://qudt.org/vocab/unit/EUR-PER-KiloW, None)]                                                      79
                                                                                                                 ... 
[(kWh, 1.0, http://qudt.org/vocab/unit/KiloW-HR, None), (d, -1.0, http://qudt.org/vocab/unit/DAY, None)]            1
[(tonnes, 1.0, http://qudt.org/vocab/unit/TONNE, None), (day, -1.0, http://qudt.org/vocab/unit/DAY, None)]          1
[(US$, 1.0, http://qudt.org/vocab/curren

In [ ]:
#target_unit = [("GW", 1.0, "http://qudt.org/vocab/unit/MegaW", None)]
target_unit = [("$", 1.0, "http://qudt.org/vocab/currency/USD", None), ("kWh", -1.0, "http://qudt.org/vocab/unit/KiloW-HR", None)]
technologies_df['value_converted'] = technologies_df.apply(
   lambda row: visualisation.convert_row(row, target_unit), axis=1
   )

In [ ]:
technologies_df = technologies_df[technologies_df["value_converted"].notna()]
technologies_df = technologies_df.reset_index(drop=True)
technologies_df.to_excel("../data/visualisation_files/unit_converted/df_technologies_capex.xlsx")